# Correction — P1 C4

Objectif : entraîner un modèle de classification avec validation croisée, en évitant les erreurs classiques du notebook initial.

Corrections principales :

- définition des variables `REGRESSION_TARGET` et `CLASSIFICATION_TARGET` ;
- chargement robuste des fichiers `.parquet` et `.json` ;
- utilisation réelle de `features_used.json` ;
- tri temporel avant `TimeSeriesSplit` ;
- conversion propre Polars → NumPy ;
- vérification des colonnes manquantes ;
- fonction de cross-validation plus lisible ;
- ajout d'un modèle `XGBClassifier` prêt à évaluer.

In [1]:
# Si xgboost n'est pas installé dans ton environnement uv, lance dans le terminal :
# uv add xgboost

from pathlib import Path
import json

import numpy as np
import polars as pl

from sklearn.model_selection import cross_validate, TimeSeriesSplit
from xgboost import XGBClassifier

In [2]:
# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

REGRESSION_TARGET = "prix"
CLASSIFICATION_TARGET = "en_dessous_du_marche"

SELECTED_REGION = "nom_region_Occitanie"

RANDOM_STATE = 42

# Racine du projet.
# Si le notebook est dans /exercices, on remonte à la racine.
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "exercices":
    ROOT_DIR = CURRENT_DIR.parent
else:
    ROOT_DIR = CURRENT_DIR

print("Dossier courant :", CURRENT_DIR)
print("Racine projet   :", ROOT_DIR)

Dossier courant : /Users/vincentdesmouceaux/exo2ml/exercices
Racine projet   : /Users/vincentdesmouceaux/exo2ml


In [3]:
# ------------------------------------------------------------
# 2. Fonctions utilitaires pour trouver les fichiers
# ------------------------------------------------------------

def find_file(filename: str, root: Path = ROOT_DIR) -> Path:
    """
    Recherche un fichier dans le projet.

    Cela évite les erreurs de chemin lorsque le notebook est lancé
    depuis PyCharm, Jupyter Lab ou un autre dossier.
    """
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Impossible de trouver le fichier : {filename}\n"
            f"Dossier de recherche : {root}"
        )

    return matches[0]


parquet_path = find_file("transactions_post_feature_engineering.parquet")
features_path = find_file("features_used.json")

print("Fichier parquet :", parquet_path)
print("Fichier features :", features_path)

Fichier parquet : /Users/vincentdesmouceaux/exo2ml/data/donnees_maitrisez_apprentissage_supervise/transactions_post_feature_engineering.parquet
Fichier features : /Users/vincentdesmouceaux/exo2ml/screencasts/features_used.json


In [4]:
# ------------------------------------------------------------
# 3. Chargement des données
# ------------------------------------------------------------

transactions = pl.read_parquet(parquet_path)

with open(features_path, "r", encoding="utf-8") as f:
    feature_names = json.load(f)

print("Shape transactions :", transactions.shape)
print("Nombre de features dans features_used.json :", len(feature_names))
print("Exemples de features :", feature_names[:10])

Shape transactions : (449172, 38)
Nombre de features dans features_used.json : 19
Exemples de features : ['type_batiment_Appartement', 'vefa', 'surface_habitable', 'latitude', 'longitude', 'mois_transaction', 'annee_transaction', 'nom_region_Auvergne-Rhône-Alpes', 'nom_region_Nouvelle-Aquitaine', 'nom_region_Occitanie']


In [5]:
# ------------------------------------------------------------
# 4. Vérification des colonnes nécessaires
# ------------------------------------------------------------

required_columns = set(feature_names + [REGRESSION_TARGET, CLASSIFICATION_TARGET, SELECTED_REGION])

missing_columns = sorted(required_columns - set(transactions.columns))

if missing_columns:
    raise ValueError(
        "Certaines colonnes nécessaires sont absentes du dataset :\n"
        + "\n".join(f"- {col}" for col in missing_columns)
    )

print("Toutes les colonnes nécessaires sont disponibles.")

Toutes les colonnes nécessaires sont disponibles.


In [6]:
# ------------------------------------------------------------
# 5. Filtrage sur la région choisie
# ------------------------------------------------------------

region_transactions = transactions.filter(
    pl.col(SELECTED_REGION) == 1
)

print("Shape après filtrage région :", region_transactions.shape)

if region_transactions.height == 0:
    raise ValueError(
        f"Aucune ligne trouvée pour la région : {SELECTED_REGION}. "
        "Vérifie que cette colonne existe bien et contient des 1."
    )

Shape après filtrage région : (50982, 38)


In [7]:
# ------------------------------------------------------------
# 6. Tri temporel avant TimeSeriesSplit
# ------------------------------------------------------------
# TimeSeriesSplit découpe les données dans l'ordre où elles sont fournies.
# Il faut donc trier les transactions chronologiquement si les colonnes existent.

sort_columns = [
    col for col in ["annee_transaction", "mois_transaction"]
    if col in region_transactions.columns
]

if sort_columns:
    region_transactions = region_transactions.sort(sort_columns)
    print("Données triées selon :", sort_columns)
else:
    print(
        "Aucune colonne temporelle trouvée pour le tri. "
        "TimeSeriesSplit utilisera l'ordre actuel des lignes."
    )

Données triées selon : ['annee_transaction', 'mois_transaction']


In [8]:
# ------------------------------------------------------------
# 7. Construction de X et y
# ------------------------------------------------------------
# X contient uniquement les features validées dans features_used.json.
# y contient la cible de classification.

model_columns = feature_names + [CLASSIFICATION_TARGET]

model_data = (
    region_transactions
    .select(model_columns)
    .drop_nulls()
)

X = model_data.select(feature_names)
y_classification = model_data[CLASSIFICATION_TARGET]

# XGBoost attend généralement une cible numérique.
# Si la cible est booléenne, on la convertit en 0 / 1.
if y_classification.dtype == pl.Boolean:
    y_classification = y_classification.cast(pl.Int8)

print("Shape X :", X.shape)
print("Shape y :", y_classification.shape)

print("Répartition de la cible :")
print(y_classification.value_counts())

Shape X : (50982, 19)
Shape y : (50982,)
Répartition de la cible :
shape: (2, 2)
┌──────────────────────┬───────┐
│ en_dessous_du_marche ┆ count │
│ ---                  ┆ ---   │
│ i32                  ┆ u32   │
╞══════════════════════╪═══════╡
│ 1                    ┆ 21186 │
│ 0                    ┆ 29796 │
└──────────────────────┴───────┘


In [9]:
# ------------------------------------------------------------
# 8. Définition du modèle XGBoost
# ------------------------------------------------------------

model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

In [10]:
# ------------------------------------------------------------
# 9. Fonction de validation croisée
# ------------------------------------------------------------

def perform_cross_validation(
    X: pl.DataFrame,
    y: pl.Series,
    model,
    cross_val_type,
    scoring_metrics: tuple[str, ...],
    return_estimator: bool = False,
    groups=None,
):
    """
    Entraîne et évalue un modèle avec validation croisée.

    Paramètres
    ----------
    X : pl.DataFrame
        Variables explicatives.

    y : pl.Series
        Variable cible.

    model :
        Modèle Scikit-learn ou compatible Scikit-learn.

    cross_val_type :
        Stratégie de validation croisée, par exemple TimeSeriesSplit.

    scoring_metrics : tuple
        Métriques à calculer : accuracy, precision, recall, f1, roc_auc...

    return_estimator : bool
        Si True, renvoie aussi les modèles entraînés pour chaque fold.

    groups :
        Groupes éventuels, utile pour GroupKFold.
    """

    scores = cross_validate(
        estimator=model,
        X=X.to_numpy(),
        y=y.to_numpy(),
        cv=cross_val_type,
        return_train_score=True,
        return_estimator=return_estimator,
        scoring=scoring_metrics,
        groups=groups,
        error_score="raise",
    )

    print("\n==============================")
    print("Résultats de cross-validation")
    print("==============================")

    for metric in scoring_metrics:
        train_values = scores["train_" + metric]
        test_values = scores["test_" + metric]

        print(f"\nMétrique : {metric}")
        print(f"Average Train {metric} : {np.mean(train_values):.4f}")
        print(f"Train {metric} Standard Deviation : {np.std(train_values):.4f}")
        print(f"Average Test {metric} : {np.mean(test_values):.4f}")
        print(f"Test {metric} Standard Deviation : {np.std(test_values):.4f}")

    return scores

In [11]:
# ------------------------------------------------------------
# 10. Exécution de la validation croisée temporelle
# ------------------------------------------------------------
# TimeSeriesSplit respecte l'ordre temporel :
# les folds d'entraînement se situent avant les folds de test.
# C'est plus cohérent si on veut simuler une prédiction sur des données futures.

time_series_cv = TimeSeriesSplit(n_splits=5)

scoring_metrics = (
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
)

scores = perform_cross_validation(
    X=X,
    y=y_classification,
    model=model,
    cross_val_type=time_series_cv,
    scoring_metrics=scoring_metrics,
)


Résultats de cross-validation

Métrique : accuracy
Average Train accuracy : 0.7117
Train accuracy Standard Deviation : 0.0099
Average Test accuracy : 0.6853
Test accuracy Standard Deviation : 0.0122

Métrique : precision
Average Train precision : 0.6867
Train precision Standard Deviation : 0.0088
Average Test precision : 0.6467
Test precision Standard Deviation : 0.0092

Métrique : recall
Average Train recall : 0.5885
Train recall Standard Deviation : 0.0384
Average Test recall : 0.5210
Test recall Standard Deviation : 0.0809

Métrique : f1
Average Train f1 : 0.6333
Train f1 Standard Deviation : 0.0254
Average Test f1 : 0.5736
Test f1 Standard Deviation : 0.0504

Métrique : roc_auc
Average Train roc_auc : 0.7857
Train roc_auc Standard Deviation : 0.0147
Average Test roc_auc : 0.7443
Test roc_auc Standard Deviation : 0.0165


## Interprétation attendue

Pour interpréter les résultats, il faut comparer les scores de train et de test.

- Si les scores de train sont très hauts mais les scores de test nettement plus faibles, le modèle fait probablement de l'overfitting.
- Si les scores de train et de test sont tous les deux faibles, le modèle est probablement en underfitting.
- Si les scores de test sont proches des scores de train, le modèle généralise mieux.

Pour un problème de classification, il ne faut pas regarder uniquement l'`accuracy`.

Dans le cas d'une classe minoritaire, le `recall`, la `precision`, le `f1-score` et le `roc_auc` sont souvent plus informatifs.

## Pourquoi cette correction est plus propre ?

Le notebook initial contenait plusieurs problèmes :

1. `REGRESSION_TARGET` et `CLASSIFICATION_TARGET` étaient utilisés sans être définis.
2. `features_used.json` était chargé, mais la liste `feature_names` n'était pas utilisée pour construire `X`.
3. Le chemin du fichier `.parquet` était fixe, donc fragile selon l'endroit depuis lequel le notebook est lancé.
4. `TimeSeriesSplit` était importé, mais les données n'étaient pas triées avant son utilisation.
5. Aucun modèle n'était réellement instancié ni évalué dans l'énoncé.
6. La fonction de validation croisée affichait les scores, mais sans contexte pédagogique.

Cette version corrige ces points et rend le notebook directement exécutable.